In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
!mkdir -p /root/.kaggle 

In [3]:
! mv kaggle.json /root/.kaggle

In [4]:
! kaggle competitions download -c dogs-vs-cats

 99% 269M/271M [00:05<00:00, 80.3MB/s]
100% 271M/271M [00:05<00:00, 53.8MB/s]
  0% 0.00/86.8k [00:00<?, ?B/s]
100% 86.8k/86.8k [00:00<00:00, 28.5MB/s]
100% 543M/543M [00:05<00:00, 162MB/s]
100% 543M/543M [00:05<00:00, 95.4MB/s]


In [5]:
! unzip -q train.zip

In [7]:
import os


In [8]:
imagenames=[]
labels=[]
for imageName in os.listdir("train/"):
  imagenames.append(imageName)
  if "cat" in imageName: 
    labels.append('cat')
  elif 'dog' in imageName:
    labels.append('dog')
  else:
    print(imageName)

In [9]:
df_train=pd.DataFrame()
df_train['Imagename']=imagenames
df_train['labels'] =labels

In [10]:
df_train.head()

,Imagename,labels
0,dog.3296.jpg,dog
1,dog.12249.jpg,dog
2,cat.10217.jpg,cat
3,dog.3939.jpg,dog
4,dog.3339.jpg,dog


In [30]:
idg =tf.keras.preprocessing.image.ImageDataGenerator(horizontal_flip=True,rotation_range=30,
                     width_shift_range=0.1,
                     height_shift_range=0.1,rescale=1/255.0,
                     zoom_range=(0,0.1),validation_split=0.1)

In [31]:
bs=64

In [32]:
train_gen=idg.flow_from_dataframe(df_train,
                                  directory="train",
                                  x_col='Imagename',
                                  y_col='labels',
                                  batch_size=bs,
                                  subset="training",
                                  target_size=(150,150))

Found 22500 validated image filenames belonging to 2 classes.


In [33]:
val_gen=idg.flow_from_dataframe(df_train,directory="train",
                                x_col='Imagename',y_col='labels',
                                batch_size=bs,subset="validation",
                                target_size=(150,150))

Found 2500 validated image filenames belonging to 2 classes.


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Conv2D, MaxPool2D, Input
from tensorflow.keras.activations import relu, softmax
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.losses import categorical_crossentropy

In [35]:
#del model
model = Sequential()

In [36]:
model.add(Input(shape=(150, 150, 3), name = "InpLayer"))
#chunk1
model.add(Conv2D(filters=16, kernel_size=(3,3), strides=(1,1), activation=relu))
model.add(MaxPool2D(pool_size=(2,2), strides=(2,2)))
#chunk2
model.add(Conv2D(filters=32, kernel_size=(3,3), strides=(1,1), activation=relu))
model.add(MaxPool2D(pool_size=(2,2), strides=(2,2)))


model.add(Flatten())
model.add(Dense(units=64, activation=relu))
model.add(Dense(units=2, activation=softmax))
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_2 (Conv2D)           (None, 148, 148, 16)      448       
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 74, 74, 16)       0         
 2D)                                                             
                                                                 
 conv2d_3 (Conv2D)           (None, 72, 72, 32)        4640      
                                                                 
 max_pooling2d_3 (MaxPooling  (None, 36, 36, 32)       0         
 2D)                                                             
                                                                 
 flatten_1 (Flatten)         (None, 41472)             0         
                                                                 
 dense_2 (Dense)             (None, 64)               

In [37]:
model.compile(optimizer=SGD(),loss=categorical_crossentropy, metrics=['acc'])

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist.history.get("acc"), label="Accuracy")
plt.plot(hist.history.get("val_acc"), label="Val Acc")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist.history.get("loss"), label="Loss")
plt.plot(hist.history.get("val_loss"), label="Val Loss")
plt.legend()
plt.show()

In [ ]:
hist = model.fit(train_gen, batch_size=bs, epochs=25, validation_data=val_gen)

Epoch 1/25
352/352 [==============================] - 193s 547ms/step - loss: 0.6952 - acc: 0.4986 - val_loss: 0.6949 - val_acc: 0.4672
Epoch 2/25
352/352 [==============================] - 185s 526ms/step - loss: 0.6924 - acc: 0.5084 - val_loss: 0.6906 - val_acc: 0.5432
Epoch 3/25
352/352 [==============================] - 186s 528ms/step - loss: 0.6914 - acc: 0.5198 - val_loss: 0.6891 - val_acc: 0.5280
Epoch 4/25
352/352 [==============================] - 187s 530ms/step - loss: 0.6905 - acc: 0.5189 - val_loss: 0.6881 - val_acc: 0.5360
Epoch 5/25
352/352 [==============================] - 186s 528ms/step - loss: 0.6888 - acc: 0.5333 - val_loss: 0.6835 - val_acc: 0.5576
Epoch 6/25
352/352 [==============================] - 189s 536ms/step - loss: 0.6875 - acc: 0.5479 - val_loss: 0.6870 - val_acc: 0.5680
Epoch 7/25
352/352 [==============================] - 185s 527ms/step - loss: 0.6859 - acc: 0.5573 - val_loss: 0.6810 - val_acc: 0.5796
Epoch 8/25
352/352 [============================

(150, 150)